In [0]:
# ── Gold: player_season_stats ────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql import Window

app     = spark.table("football_catalog.silver.fact_appearances")
games   = spark.table("football_catalog.silver.fact_games").select("game_id","season","competition_id")
players = spark.table("football_catalog.silver.dim_players").filter("is_current = True").select("player_id","player_name","position","nationality")

gold = (app
    .join(games,   "game_id")
    .join(players, "player_id")
    .groupBy("player_id","player_name","position","nationality","season","competition_id")
    .agg(
        F.count("appearance_id").alias("appearances"),
        F.sum("goals").alias("total_goals"),
        F.sum("assists").alias("total_assists"),
        F.sum("yellow_cards").alias("yellow_cards"),
        F.sum("red_cards").alias("red_cards"),
        F.sum("minutes_played").alias("total_minutes"),
        F.round(F.sum("goals")/F.count("appearance_id"),2).alias("goals_per_game")
    )
)

# Rank goal scorers within each competition per season
w = Window.partitionBy("competition_id","season").orderBy(F.desc("total_goals"))
gold_final = gold.withColumn("goals_rank", F.rank().over(w))

(gold_final.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("football_catalog.gold.player_season_stats"))

print("SUCCESS: Gold player_season_stats created")
spark.table("football_catalog.gold.player_season_stats").show(5)